In [1]:
import os
import requests
import xml.etree.ElementTree as ET
import xml.dom.minidom

from datetime import datetime
from bs4 import BeautifulSoup

import pandas as pd

In [2]:
def get_df(sheet_id ,gid):
    csv_url = (
        f"https://docs.google.com/spreadsheets/d/{sheet_id}/export"
        f"?format=csv&id={sheet_id}&gid={gid}"
    )

    return pd.read_csv(csv_url)

In [3]:
xml_df = get_df("1p7DlFDqk5RXT1X9ijevAVTMLepMki8pFHacYmYaVKrQ","0")
encounters_df = get_df("1p7DlFDqk5RXT1X9ijevAVTMLepMki8pFHacYmYaVKrQ","327410979")
enenra_df = get_df("1RvvRFdEfbi7Nzwz26fazfZNwQ0hE5UWYnrs02175HBo","230306071")
enenra_df = (
    enenra_df
    .drop(enenra_df.index[0])
    .dropna()
    .reset_index(drop=True)
)
int_cols = [
    'MinPriceMultiplier',
    'MaxPriceMultiplier',
    'MinAmount',
    'MaxAmount',
]

for col in int_cols:
    bad = enenra_df.loc[
        pd.to_numeric(enenra_df[col], errors="coerce").isna(),
        col
    ].unique()
    print(col, bad)

    
enenra_df[int_cols] = enenra_df[int_cols].astype(int)

contracts_df = get_df("1RvvRFdEfbi7Nzwz26fazfZNwQ0hE5UWYnrs02175HBo","1601229222")
contract_cols = [
    "MissionIds",
    "ITC",
    "SHIVAN",
    "AGURO",
    "SOLCOOP",
    "ZENOVA",
    "MAYOR",
    "SECURITY"
]

# Rename MissionIds to the desired contract block name
contracts_df = (
    contracts_df.melt(
        id_vars=["Faction_Name"],
        value_vars=contract_cols,
        var_name="ContractBlockType",
        value_name="MissionId"
    )
    .dropna(subset=["MissionId"])
)

contracts_df["ContractBlockType"] = contracts_df["ContractBlockType"].replace({
    "MissionIds": "DEFAULT"
})

# Split comma-separated mission IDs into rows
contracts_df["MissionId"] = contracts_df["MissionId"].str.split(",")

contracts_df = contracts_df.explode("MissionId")

# Clean whitespace
contracts_df["MissionId"] = contracts_df["MissionId"].str.strip()

contracts_df = contracts_df.reset_index(drop=True)

MinPriceMultiplier []
MaxPriceMultiplier []
MinAmount []
MaxAmount []


In [4]:
def reorganize_table_to_dict(df):
    entries = {}

    for _, row in df.iterrows():
        container_name = row["StoreItemsContainer"]
        item_id = row["StoreItemId"]

        if container_name not in entries:
            entries[container_name] = {}

        if item_id not in entries[container_name]:
            entries[container_name][item_id] = {
                "ItemType": row["ItemType"],
                "ItemSubtypeIds": [],
                "MinPriceMultiplier": str(row["MinPriceMultiplier"]),
                "MaxPriceMultiplier": str(row["MaxPriceMultiplier"]),
                "MinAmount": str(row["MinAmount"]),
                "MaxAmount": str(row["MaxAmount"]),
            }

        entries[container_name][item_id]["ItemSubtypeIds"].append(
            row["ItemSubtypeIds"].split("/")[-1]
        )

    return entries

def write_dialoguebank(df):

    entries = reorganize_table_to_dict(df)

    for container_name, container_content in entries.items():
        container = ET.Element('StoreItemsContainer')
        container.set('xmlns:xsd', 'http://www.w3.org/2001/XMLSchema')
        container.set('xmlns:xsi', 'http://www.w3.org/2001/XMLSchema-instance')

        items = ET.SubElement(container, 'StoreItems')

        for id, id_content in container_content.items():
            item = ET.SubElement(items, 'StoreItem')

            item_id = ET.SubElement(item, 'StoreItemId')
            item_id.text = id

            item_type = ET.SubElement(item, 'ItemType')
            item_type.text = id_content["ItemType"]

            item_subtypes = ET.SubElement(item, 'ItemSubtypeIds')

            for item_subtypeid in id_content["ItemSubtypeIds"]:
                item_subtype = ET.SubElement(item_subtypes, 'ItemSubtypeId')
                item_subtype.text = item_subtypeid

            offer = ET.SubElement(item, 'Offer')
            offer_minprice = ET.SubElement(offer, 'MinPriceMultiplier')
            offer_minprice.text = id_content["MinPriceMultiplier"]
            offer_maxprice = ET.SubElement(offer, 'MaxPriceMultiplier')
            offer_maxprice.text = id_content["MaxPriceMultiplier"]
            offer_minamount = ET.SubElement(offer, 'MinAmount')
            offer_minamount.text = id_content["MinAmount"]
            offer_maxamount = ET.SubElement(offer, 'MaxAmount')
            offer_maxamount.text = id_content["MaxAmount"]

            order = ET.SubElement(item, 'Order')
            order_minprice = ET.SubElement(order, 'MinPriceMultiplier')
            order_minprice.text = id_content["MinPriceMultiplier"]
            order_maxprice = ET.SubElement(order, 'MaxPriceMultiplier')
            order_maxprice.text = id_content["MaxPriceMultiplier"]
            order_minamount = ET.SubElement(order, 'MinAmount')
            order_minamount.text = id_content["MinAmount"]
            order_maxamount = ET.SubElement(order, 'MaxAmount')
            order_maxamount.text = id_content["MaxAmount"]


        temp_string = ET.tostring(container, 'utf-16')
        temp_string.decode('utf-16')
        xml_string = xml.dom.minidom.parseString(temp_string)
        xml_formatted = xml_string.toprettyxml()
        xml_formatted = xml_formatted.replace('<?xml version="1.0" ?>', f'<?xml version="1.0" encoding="utf-16"?>\n\n<!-- Created from GSheet export on {datetime.now()} -->\n')
        xml_formatted = xml_formatted.replace("<StoreItems>","<StoreItems>\n")
        xml_formatted = xml_formatted.replace("</StoreItem>","</StoreItem>\n")

        target_file = f"D:/SEMods-Github/Ares-at-War-part-1/Ares at War part 1/Data/StoreItems/{container_name}.xml"
        exported_xml = open(target_file, "w")
        exported_xml.write(xml_formatted)




In [5]:
write_dialoguebank(enenra_df)

In [6]:
enenra_df

,StoreItemsContainer,StoreblockId,Type,StoreItemId,ItemType,ItemSubtypeIds,Required,OutputToggle,MinPriceMultiplier,MaxPriceMultiplier,MinAmount,MaxAmount
0,CIVILIAN_Bulnes,Mine,Normal,OutputOres,Ore,Ore/Gold,True,True,120,125,50000,70000
1,CIVILIAN_Bulnes,Mine,Normal,OutputOres,Ore,Ore/Iron,True,True,120,125,50000,70000
2,CIVILIAN_Bulnes,Mine,Normal,OutputOres,Ore,Ore/Cobalt,True,True,120,125,50000,70000
3,CIVILIAN_Bulnes,Mine,Normal,OutputOres,Ore,Ore/Nickel,True,True,120,125,50000,70000
4,CIVILIAN_Bulnes,Mine,Normal,InputKits,Consumable,Consumable/Medkit,False,False,105,110,5,15
...,...,...,...,...,...,...,...,...,...,...,...,...
254,CIVILIAN_Azuris,Wine,Normal,InputFood,Consumable,Consumable/MealPack_Lasagna,False,False,15,20,15,20
255,CIVILIAN_Azuris,Wine,Normal,InputFood,Consumable,Consumable/MealPack_Burrito,False,False,15,20,15,20
256,CIVILIAN_Azuris,Wine,Normal,InputFood,Consumable,Consumable/MealPack_SteakDinner,False,False,15,20,15,20
257,CIVILIAN_Azuris,Wine,Normal,OutputConsumable,Consumable,Consumable/ClangCola,False,True,10,15,200,250


In [7]:
def generate_store_tags(df):
    lines = []

    for _, row in df.iterrows():

        item = row["ItemSubtypeIds"]

        # Required
        if row["Required"]:

            if row["OutputToggle"]:
                lines.append(f"[RequiredOffers:{item}]")
                continue

            if row["OutputToggle"] == False:
                lines.append(f"[RequiredOrders:{item}]")
                continue

        # Offers
        if row["OutputToggle"]:

            lines.extend([
                f"[Offers:{item}]",
            ])
            continue

        # Orders
        if row["OutputToggle"] == False:

            lines.extend([
                f"[Orders:{item}]",

            ])
            continue

    return "\n".join(lines)


def generate_Mission_tags(df):
    lines = []

    for _, row in df.iterrows():
        item = row["MissionId"]
        lines.append(f"[MissionIds:AaW_Mission_{item}]")

    return "\n".join(lines)


In [8]:
entity_components = []

for Faction_Name, group_df in contracts_df.groupby(["Faction_Name"]):

    faction, encounter = Faction_Name[0].split("_", 1)

    for ContractBlockType, group_df2 in group_df.groupby(["ContractBlockType"]):

        subtype = (
            f"{faction}_ContractBlockProfile_{encounter}"
            if ContractBlockType[0] == "DEFAULT"
            else f"{faction}_ContractBlockProfile_{encounter}_{ContractBlockType[0]}"
        )

        entity_components.append(f"""    <EntityComponent xsi:type="MyObjectBuilder_InventoryComponentDefinition">
      <Id>
        <TypeId>Inventory</TypeId>
        <SubtypeId>{subtype}</SubtypeId>
      </Id>
      <Description>
[MES Contract Block]
[StoreProfileId:{faction}_StoreProfile_{encounter}_Ingot]

[MinContracts:5]
[MaxContracts:15]
{generate_Mission_tags(group_df)}
      </Description>
    </EntityComponent>""")

with open("ContractBlockProfiles.sbc", "w", encoding="utf-8") as f:
    f.write(
        '<?xml version="1.0"?>\n'
        '<Definitions xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" '
        'xmlns:xsd="http://www.w3.org/2001/XMLSchema">\n'
        '  <EntityComponents>\n\n'
        + "\n\n".join(entity_components)
        + '\n\n  </EntityComponents>\n'
        '</Definitions>\n'
    )

In [9]:
store_profiles = []

for store_items_container, group_df in enenra_df.groupby(["StoreItemsContainer"]):

    faction, encounter = store_items_container[0].split("_", 1)

    for StoreblockId, group_df2 in group_df.groupby(["StoreblockId"]):

        store_profiles.append(f"""    <EntityComponent xsi:type="MyObjectBuilder_InventoryComponentDefinition">
      <Id>
        <TypeId>Inventory</TypeId>
        <SubtypeId>{faction}_StoreProfile_{encounter}_{StoreblockId[0]}</SubtypeId>
      </Id>
      <Description>
[MES Store]

[FileSource:{store_items_container[0]}.xml]

[MinOfferItems:99]
[MaxOfferItems:100]
[MinOrderItems:99]
[MaxOrderItems:100]

[ItemsRequireInventory:false]

{generate_store_tags(group_df2)}
      </Description>
    </EntityComponent>\n\n""")

# Build SBC content
sbc_content = f"""<?xml version="1.0"?>
<Definitions xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema">
  <EntityComponents>

{"".join(store_profiles)}

  </EntityComponents>
</Definitions>
"""

# Export file
with open("StoreProfiles.sbc", "w", encoding="utf-8") as f:
    f.write(sbc_content)

print(f"Created StoreProfiles.sbc with {len(store_profiles)} store profiles.")

Created StoreProfiles.sbc with 10 store profiles.


In [10]:
contracts_df

,Faction_Name,ContractBlockType,MissionId
0,CIVILIAN_Azuris,DEFAULT,Azuris_DestroyDRAMagnesiumOther
1,CIVILIAN_Azuris,DEFAULT,ProcAZURIS
2,CIVILIAN_Azuris,DEFAULT,ProcAZURIS
3,CIVILIAN_Azuris,DEFAULT,UNION_NextSteps
4,CIVILIAN_Azuris,DEFAULT,Azuris_LaunchUNION
5,CIVILIAN_Azuris,DEFAULT,MILITIA_PersuadeAzuris
6,CIVILIAN_SunsetCity,DEFAULT,SunsetCity_DestroySHIVANRavun
7,CIVILIAN_SunsetCity,DEFAULT,SunsetCity_LaunchUNION
8,CIVILIAN_SunsetCity,DEFAULT,ProcCIVILIAN
9,CIVILIAN_SunsetCity,DEFAULT,ProcCIVILIAN


In [11]:
contracts_df.loc[
    contracts_df["Faction_Name"] == "CIVILIAN_Azuris",
    "ContractBlockType"
].unique().tolist()

['DEFAULT']

In [12]:
def GetContractInfo(store_items_container):
    Faction, Name = store_items_container[0].split("_", 1)
    tja = contracts_df.loc[contracts_df["Faction_Name"] == store_items_container[0],"ContractBlockType"].unique().tolist()

    Contracts = ""

    for type in tja:
        if type == "DEFAULT":
           Contracts += f"""[ContractBlocks:Contracts]
[ContractBlockProfiles:{Faction}_ContractBlockProfile_{Name}]\n"""
           continue

        Contracts += f"""[ContractBlocks:Contracts {type}]
[ContractBlockProfiles:{Faction}_ContractBlockProfile_{Name}_{type}]\n"""
        
    return Contracts




for store_items_container, group_df in enenra_df.groupby(["StoreItemsContainer"]):
    Faction, Name = store_items_container[0].split("_", 1)
    StoreBlocks = ""

    for StoreblockId, group_df2 in group_df.groupby(["StoreblockId"]):
        StoreblockId = StoreblockId[0]
        StoreBlocks += f"""[StoreBlocks:Store {StoreblockId}]
[StoreProfiles:{Faction}_StoreProfile_{Name}_{StoreblockId}]"""


    print(f"""<EntityComponent xsi:type="MyObjectBuilder_InventoryComponentDefinition">
<Id>
    <TypeId>Inventory</TypeId>
    <SubtypeId>{Faction}_Trigger_Static_PopulateStores_{Name}</SubtypeId>
</Id>
<Description>

[RivalAI Trigger]
[Tags:StoreRefresh]
[UseTrigger:true]
[Type:Timer]
[MinCooldownMs:1200000]
[MaxCooldownMs:1200001]
[StartsReady:true]
[MaxActions:-1]
[Actions:{Faction}_Action_Static_PopulateStores_{Name}]

</Description>
</EntityComponent>

<EntityComponent xsi:type="MyObjectBuilder_InventoryComponentDefinition">
<Id>
    <TypeId>Inventory</TypeId>
    <SubtypeId>{Faction}_Action_Static_PopulateStores_{Name}</SubtypeId>
</Id>
<Description>
[RivalAI Action]

[ApplyStoreProfiles:true]
[ClearStoreContentsFirst:true]

{StoreBlocks}

[StoreBlocks:Store Fuel]
[StoreProfiles:{Faction}_StoreProfile_{Name}_Fuel]

[AddCustomDataToBlocks:true]
[CustomDataBlockNames:EconomySurvival Store Water Vehicles,EconomySurvival Store Land Vehicles,EconomySurvival Store Air Vehicles,EconomySurvival Store Space Vehicles]
[CustomDataFiles:{Faction}_Water_Vehicles.xml]
[CustomDataFiles:{Faction}_Land_Vehicles.xml]
[CustomDataFiles:{Faction}_Air_Vehicles.xml]
[CustomDataFiles:{Faction}_Space_Vehicles.xml]  


[ApplyContractProfiles:true]
[ClearContractContentsFirst:true]
{GetContractInfo(store_items_container)}

</Description>
</EntityComponent>""")




<EntityComponent xsi:type="MyObjectBuilder_InventoryComponentDefinition">
<Id>
    <TypeId>Inventory</TypeId>
    <SubtypeId>CIVILIAN_Trigger_Static_PopulateStores_Azuris</SubtypeId>
</Id>
<Description>

[RivalAI Trigger]
[Tags:StoreRefresh]
[UseTrigger:true]
[Type:Timer]
[MinCooldownMs:1200000]
[MaxCooldownMs:1200001]
[StartsReady:true]
[MaxActions:-1]
[Actions:CIVILIAN_Action_Static_PopulateStores_Azuris]

</Description>
</EntityComponent>

<EntityComponent xsi:type="MyObjectBuilder_InventoryComponentDefinition">
<Id>
    <TypeId>Inventory</TypeId>
    <SubtypeId>CIVILIAN_Action_Static_PopulateStores_Azuris</SubtypeId>
</Id>
<Description>
[RivalAI Action]

[ApplyStoreProfiles:true]
[ClearStoreContentsFirst:true]

[StoreBlocks:Store Wine]
[StoreProfiles:CIVILIAN_StoreProfile_Azuris_Wine]

[StoreBlocks:Store Fuel]
[StoreProfiles:CIVILIAN_StoreProfile_Azuris_Fuel]

[AddCustomDataToBlocks:true]
[CustomDataBlockNames:EconomySurvival Store Water Vehicles,EconomySurvival Store Land Vehicles

In [13]:
trigger_components = []
trigger_ids = []

for store_items_container, group_df in enenra_df.groupby(["StoreItemsContainer"]):

    faction, name = store_items_container[0].split("_", 1)

    trigger_id = f"{faction}_Trigger_Static_PopulateStores_{name}"
    trigger_ids.append(trigger_id)

    store_blocks = ""

    for StoreblockId, group_df2 in group_df.groupby(["StoreblockId"]):
        storeblock_id = StoreblockId[0]

        store_blocks += f"""
[StoreBlocks:Store {storeblock_id}]
[StoreProfiles:{faction}_StoreProfile_{name}_{storeblock_id}]
"""

    trigger_components.append(f"""
    <EntityComponent xsi:type="MyObjectBuilder_InventoryComponentDefinition">
      <Id>
        <TypeId>Inventory</TypeId>
        <SubtypeId>{trigger_id}</SubtypeId>
      </Id>
      <Description>

[RivalAI Trigger]
[Tags:StoreRefresh]
[UseTrigger:true]
[Type:Timer]
[MinCooldownMs:1200000]
[MaxCooldownMs:1200001]
[StartsReady:true]
[MaxActions:-1]
[Actions:{faction}_Action_Static_PopulateStores_{name}]

      </Description>
    </EntityComponent>

    <EntityComponent xsi:type="MyObjectBuilder_InventoryComponentDefinition">
      <Id>
        <TypeId>Inventory</TypeId>
        <SubtypeId>{faction}_Action_Static_PopulateStores_{name}</SubtypeId>
      </Id>
      <Description>
[RivalAI Action]

[ApplyStoreProfiles:true]
[ClearStoreContentsFirst:true]

{store_blocks}

[StoreBlocks:Store Fuel]
[StoreProfiles:{faction}_StoreProfile_{name}_Fuel]

[AddCustomDataToBlocks:true]
[CustomDataBlockNames:EconomySurvival Store Water Vehicles,EconomySurvival Store Land Vehicles,EconomySurvival Store Air Vehicles,EconomySurvival Store Space Vehicles]
[CustomDataFiles:{faction}_Water_Vehicles.xml]
[CustomDataFiles:{faction}_Land_Vehicles.xml]
[CustomDataFiles:{faction}_Air_Vehicles.xml]
[CustomDataFiles:{faction}_Space_Vehicles.xml]

[ApplyContractProfiles:true]
[ClearContractContentsFirst:true]
{GetContractInfo(store_items_container)}

      </Description>
    </EntityComponent>
""")

# Write Triggers.sbc
with open("Triggers.sbc", "w", encoding="utf-8") as f:
    f.write(
        '<?xml version="1.0"?>\n'
        '<Definitions xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" '
        'xmlns:xsd="http://www.w3.org/2001/XMLSchema">\n'
        '  <EntityComponents>\n\n'
        + "\n".join(trigger_components)
        + '\n\n  </EntityComponents>\n'
        '</Definitions>\n'
    )

# Print all trigger IDs
print("=== Trigger IDs ===")
for trigger_id in trigger_ids:
    print(trigger_id)

=== Trigger IDs ===
CIVILIAN_Trigger_Static_PopulateStores_Azuris
CIVILIAN_Trigger_Static_PopulateStores_Bulnes
CIVILIAN_Trigger_Static_PopulateStores_Deba
CIVILIAN_Trigger_Static_PopulateStores_Halsa
CIVILIAN_Trigger_Static_PopulateStores_Lanes
CIVILIAN_Trigger_Static_PopulateStores_Lusarbe
CIVILIAN_Trigger_Static_PopulateStores_Orio
CIVILIAN_Trigger_Static_PopulateStores_SunsetCity
CIVILIAN_Trigger_Static_PopulateStores_Thorrix
